# Notebook image fit check

Covers [suzuri#74](https://github.com/harrywang/suzuri/pull/74): image outputs used to keep their pixel width (capped only at `repl.max_columns`, about 1000px), so a chart in a notebook pane narrower than the image was clipped on the right instead of scaled down the way Jupyter and VS Code do it. Run every cell (the testbed `.venv` kernel has matplotlib), then tick the list.

- [ ] Section 1: the ~1600px-wide chart fits the pane, with the right-most bar group and the end of the title visible
- [ ] Section 1: dragging the project panel or agent panel edge to narrow the notebook shrinks the chart live; widening it grows the chart back, never past its natural size
- [ ] Section 2: the circle stays a circle at every pane width (aspect ratio kept, no stretching)
- [ ] Section 3: the ~290×190 chart stays small; it is not blown up to the pane width
- [ ] Section 4: the copy button beside the output still copies the full-resolution image
- [ ] Section 5 (optional): with `"repl": { "output_max_height_lines": 10 }` in settings, the tall chart shrinks in both dimensions instead of being squashed


## 1. A chart wider than the pane

The same shape as the report that found the bug: grouped bars, a long title, and rotated category labels, drawn at about 1590×590 pixels so it overflows any split layout.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 100

categories = ["Electronic accessories", "Fashion accessories", "Food and beverages", "Health and beauty", "Home and lifestyle", "Sports and travel"]
shares = {
    "A": [17.2, 15.4, 16.2, 11.9, 21.1, 18.2],
    "B": [16.1, 15.5, 14.3, 18.8, 16.5, 18.8],
    "C": [17.2, 19.5, 21.5, 15.0, 13.4, 13.4],
}

x = np.arange(len(categories))
fig, ax = plt.subplots(figsize=(16, 6))
for offset, (branch, values) in zip((-0.27, 0, 0.27), shares.items()):
    ax.bar(x + offset, values, width=0.27, label=branch)
ax.set_xticks(x, categories, rotation=25, ha="right")
ax.set_ylabel("share of that branch's revenue (%)")
ax.set_title("Each branch sells a different mix: Home & lifestyle is 21% of Branch A but 13% of Branch C  <- this end must be visible")
ax.legend(title="Branch")
plt.tight_layout()
plt.show()


## 2. Aspect ratio

A figure about 1150×600 pixels with an equal-aspect circle. If the image were squeezed horizontally without its height following, the circle would turn into an ellipse.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, linewidth=4))
ax.set_aspect("equal")
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(-1.1, 1.1)
ax.set_title("This circle must stay round at every pane width")
plt.show()


## 3. A chart smaller than the pane

About 290×190 pixels. It should render at that size, not be scaled up to fill the width.

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2))
ax.plot([1, 2, 3, 4], [1, 4, 2, 3], marker="o")
ax.set_title("Small: natural size")
plt.tight_layout()
plt.show()


## 4. Copying

Click the copy button beside the section 1 chart and paste it into Preview (`File → New from Clipboard`). It should be the full ~1590×590 image, not the scaled-down on-screen size.

## 5. Output height limit (optional)

The fit is capped by `repl.output_max_height_lines` too. Set it to `10`, rerun this cell, and the chart should shrink proportionally to about ten lines tall. Remove the setting afterwards; the default is `0` (no limit).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12))
ax.barh(categories, shares["A"])
ax.set_title("Tall chart: shrinks proportionally under a height limit")
plt.tight_layout()
plt.show()
